In [1]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, "..")
from importlib import reload
import equations as eq
import trajectory_lib as tr
reload (tr);
reload (eq);

motion_list  = ['Flexion']
GH_seq = 'YZY'
weight= 102
motion_folder = motion_list[0]
motion_name = motion_list[0]

data_struct = sc.io.loadmat('../data_model.mat')
OS_struct = sc.io.loadmat('../Motions/'+motion_folder+'/OS_model.mat')

act_w = 1
vel_w = 0.1

MM,FO,q,u,fr,frstar,kindeq,xdot,first_elips_scale,elips_trans = eq.create_eoms_eul(data_struct,OS_struct,derive = 'numeric',gen_matlab_functions = 0,GH_seq = GH_seq)
TE,activations,TE_conoid = eq.polynomials_euler(OS_struct,q,derive = 'numeric',model_params_struct = data_struct)


equations created


In [2]:
traj_w = weight
struct_name = 'res_euler_'+motion_list[0]+'_'+str(weight)

In [3]:
excitations = []
act_ode = []
# print(activations)
for i in range(len(activations)):
    excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
    act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))

sp_act_ode = sp.Matrix(act_ode)

In [4]:
eoms_implicit = sp.Matrix(kindeq).col_join(fr+frstar+TE+sp.Matrix(TE_conoid)).col_join(sp_act_ode)

In [5]:
reload(eq);
num_nodes = 101
file = '../Motions/' + motion_folder + '/' + motion_name
traj_original, interval_value, time = tr.exp_trajectory_eul(file,num_nodes)
traj = tr.exp_trajectory_eul_myobj(traj_original,GH_seq)

state_symbols = tuple(q+u+activations)
num_states = len(state_symbols)
num_q = len(q)
num_u = len(u)
# specified_symbols = tuple(activations)
specified_symbols = tuple(excitations)
num_inputs = len(specified_symbols)
t = me.dynamicsymbols._t
objective_traj,objective_traj_jac = eq.custom_objective_eul(len(q),interval_value, GH_seq = GH_seq)
obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes, interval_value)
w_diff_vel = 2
w_diff_act = 2
w_diff_exc = 0.1
node1 = 0
node2 = num_nodes//2
node3 = num_nodes-1

def obj(free):
    # min_traj = traj_w * interval_value * np.sum((traj_original.flatten() - free[:10*num_nodes])**2)
    min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
    # min_traj = traj_w * (objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3]))

    min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
    min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)
    min_act_dif = w_diff_act * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))))
    min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))))

    # min_act_dif = obj_act_dif(np.ones((101,5)))
    return (min_traj + min_torque + min_vel_dif + min_act_dif + min_exc_dif).item()

def obj_grad(free):
    grad = np.zeros_like(free)
    # grad[:10*num_nodes] = traj_w * 2.0 * interval_value * (free[:10*num_nodes] - traj_original.flatten())
    grad[:num_q*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))

    grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] + w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
    grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] = w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))

    grad[num_q*num_nodes:(num_q + num_u)*num_nodes] = w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))

    # ## reach ##
    # first_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1])))
    # second_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2])))
    # third_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3])))
    # # print(np.shape(first_grad_vals))
    # grad_traj = np.zeros((num_nodes,10))
    # grad_traj[node1,:] = first_grad_vals
    # grad_traj[node2,:] = second_grad_vals
    # grad_traj[node3,:] = third_grad_vals
    # grad[:10*num_nodes] = traj_w * np.concatenate(grad_traj.T)
    # ## end reach ##

    return grad
instance_constraints = []
# for i in range(9):
#     instance_constraints.append(state_symbols[i].func(0.0)-x0t[i])

instance_constraints.append(state_symbols[-1].func(0.0)-0)

bounds1 = (0.0,1.0)
bounds = (bounds1,)*len(activations)
bndrs = dict(zip(activations,bounds))
bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
bndrs.update(bndrs_exc)
angles_bndr = {q[0]: (-60*np.pi/180, -20*np.pi/180),
               q[1]: (-0.8, 0.8),
               q[2]: (-0.3, 1),
               q[3]: (0.8, 1.5),
               q[4]: (-0.3, 1),
               q[5]: (-0.5,0.5),
            #    q[6]: (-1.7,1.7),
               q[7]: (0,90*np.pi/180)}
            #    q[8]: (-1.7,1.7)}
bndrs.update(angles_bndr)

In [6]:

start = tm.time()
prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
            num_nodes, interval_value,
            known_parameter_map={},
            instance_constraints=instance_constraints,
            bounds=bndrs,
            integration_method='midpoint'
) #               
time_to_create = tm.time() - start
print(time_to_create)
prob.add_option('max_iter',10000)
prob.add_option('limited_memory_max_history', 40)
initial_guess = np.zeros(prob.num_free)
initial_guess[:10*num_nodes] = traj_original.flatten()
time_2_solve_start = tm.time()
solution, info = prob.solve(initial_guess)
time_2_solve = tm.time() - time_2_solve_start
print(info['status_msg'])
print(info['obj_val'])
act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
objective_value = prob.obj_value
print('Objective activations: ', act_obj)



1289.049420118332

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  8055601
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    27674
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    26361
                     variables with only upper bounds:        0
Total number of equality constraints.................:    14701
Total numb

In [7]:
reload(tr)
file_name = '../Motions/'+motion_folder+'/' + struct_name + '.mat'
tr.sol2struct(solution,activations,num_q,num_u,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,True)

file_name_mot = '../Motions/'+motion_folder+'/' + struct_name + '.mot'
tr.sol2mot_eul(solution, num_nodes, len(q), time, file_name_mot)

Saved to .mat file
Saved to .mot file


In [8]:
# fig, axs = plt.subplots((num_q + num_u))
# for j in range((num_q + num_u)):
#     axs[j].plot(time,solution[j*num_nodes:(j+1)*num_nodes])
#     fig.set_figheight(30)

In [9]:
# fig, axs = plt.subplots((num_inputs))
# for j in range(num_inputs):
#     axs[j].plot(time,solution[(j + num_q + num_u)*num_nodes:(j + num_q + num_u + 1)*num_nodes],label = 'activation')
#     axs[j].plot(time,solution[(j + num_q + num_u + num_inputs)*num_nodes:(j + num_q + num_u + num_inputs + 1)*num_nodes],label = 'excitation')
#     fig.set_figheight(120)
#     # axs[j].legend()

# plt.show()

In [10]:
# fig, axes = plt.subplots((num_q+num_u+2*num_inputs), 1, sharex=True,
#                          figsize=(6.4, 0.8*(num_q+num_u+2*num_inputs)),
#                          layout='compressed')
# prob.plot_trajectories(solution, axes=axes)

In [11]:
# from opty import Problem, create_objective_function, parse_free
# import sympy as sp
# import numpy as np
# import scipy as sc
# import time as tm
# import pickle
# import sympy.physics.mechanics as me
# import sys
# sys.path.insert(0, "..")
# from importlib import reload
# import equations as eq
# reload (eq);

# initPos = 'InitPosOptQuat'

# motion_folder_list = ['Elevation','Abduction_rigged2','Abduction_rigged','Abduction','Steering']
# motion_list = ['elevation.mat','abd_rigged.mat','abd_rigged.mat','abduction.mat','steering.mat']
# weights_list = [100,150,200,250,300]

# for i in range(len(motion_folder_list)):
#     for iweight in range(len(weights_list)):
#         motion_folder = motion_folder_list[i]
#         motion_name = motion_list[i]
#         struct_name = 'results_euler_QuatInit_'+str(weights_list[iweight])
#         act_w = 1
#         traj_w = weights_list[iweight]
#         vel_w = 0.1

#         model_struct = sc.io.loadmat('../Motions/'+motion_folder+'/OS_model.mat')
#         data_struct = sc.io.loadmat('../data_model.mat')

#         start = tm.time()
#         MM,FO,q,u,fr,frstar,kindeq,xdot = eq.create_eoms_eul(model_struct,data_struct,initPos,derive = 'numeric',gen_matlab_functions = 0)
#         TE,activations,TE_conoid = eq.polynomials_euler(model_struct,q,derive = 'numeric',model_params_struct = data_struct,initCond_name = initPos)

#         time_to_create = tm.time() - start
#         print(time_to_create)

#         dict_vals,symlist, value_list = eq.create_parameters_dict(data_struct, initPos)
#         x0 = data_struct['params'][initPos][0,0]['initCondEul'].item()
#         x0t = list(x0.T[0])
#         eoms_implicit = sp.Matrix(kindeq).col_join(fr+frstar+TE)
#         import trajectory_lib as tr
#         reload (tr);
#         num_nodes = 101
#         file = '../Motions/' + motion_folder + '/' + motion_name
#         traj_original, interval_value, time = tr.exp_trajectory_eul(file,num_nodes)
#         traj = tr.exp_trajectory_eul_myobj(traj_original)

#         state_symbols = tuple(q+u)
#         num_states = len(state_symbols)
#         specified_symbols = tuple(activations)
#         num_inputs = len(specified_symbols)
#         t = me.dynamicsymbols._t
#         objective_traj,objective_traj_jac = eq.custom_objective_eul(len(q),interval_value)

#         def obj(free):
#             # min_traj = traj_w * interval_value * np.sum((traj - free[:10*num_nodes])**2)
#             min_traj = traj_w * np.sum(objective_traj(np.split(free[:10*num_nodes],10),traj))
#             min_vel = vel_w * interval_value * np.sum((free[10*num_nodes:num_states*num_nodes])**2)
#             min_torque = act_w * interval_value * np.sum(free[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
#             return min_traj + min_torque + min_vel

#         def obj_grad(free):
#             grad = np.zeros_like(free)
#             # grad[:10*num_nodes] = traj_w * 2.0 * interval_value * (free[:10*num_nodes] - traj)
#             grad[:10*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:10*num_nodes],10),traj))
#             grad[10*num_nodes:num_states*num_nodes] = vel_w * 2 * interval_value * free[10*num_nodes:num_states*num_nodes]
#             grad[num_states*num_nodes:(num_states + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[num_states*num_nodes:(num_states + num_inputs)*num_nodes]
#             return grad
#         instance_constraints = []
#         # for i in range(9):
#         #     instance_constraints.append(state_symbols[i].func(0.0)-x0t[i])

#         instance_constraints.append(state_symbols[-1].func(0.0)-0)

#         bounds1 = (0.0,1.0)
#         bounds = (bounds1,)*len(activations)
#         bndrs = dict(zip(activations,bounds))
#         start = tm.time()

#         prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
#                     num_nodes, interval_value,
#                     known_parameter_map={},
#                     instance_constraints=instance_constraints,
#                     bounds=bndrs,
#                     integration_method='midpoint'
#         ) #               
#         time_to_create = tm.time() - start
#         print(time_to_create)
#         prob.add_option('max_iter',2500)
#         prob.add_option('limited_memory_max_history', 40)
#         initial_guess = np.zeros(prob.num_free)
#         initial_guess[:10*num_nodes] = traj_original.flatten()
#         time_2_solve_start = tm.time()
#         solution, info = prob.solve(initial_guess)
#         time_2_solve = tm.time() - time_2_solve_start
#         print(info['status_msg'])
#         print(info['obj_val'])
#         act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)


#         print('Objective activations: ', act_obj)
#         import matplotlib.pyplot as plt
#         fig, axes = plt.subplots(int(num_states+num_inputs), 1, sharex=True,
#                                 figsize=(6.4, 0.8*(num_states+num_inputs)),
#                                 layout='compressed')
#         prob.plot_trajectories(solution, axes=axes)
#         import matplotlib.pyplot as plt
#         fig, axs = plt.subplots(10)
#         for j in range(10):
#             axs[j].plot(time,traj_original.flatten()[j*num_nodes:(j+1)*num_nodes])
#             axs[j].plot(time,solution[j*num_nodes:(j+1)*num_nodes])
#             fig.set_figheight(10)
#         fig, axes = plt.subplots(2, figsize=(12.8, 9.6),
#                                 layout='constrained')
#         prob.plot_constraint_violations(solution, axes=axes)
#         import trajectory_lib as tr
#         reload (tr);
#         # num_iter_sol = int(input('enterr number of iterarions:'))
#         file_name = '../Motions/'+motion_folder+'/' + struct_name + '.mat'
#         tr.sol2struct(solution,activations,len(q),num_states,num_nodes,time,0,time_2_solve,file_name)

#         file_name_mot = '../Motions/'+motion_folder+'/' + struct_name + '.mot'
#         tr.sol2mot_eul(solution, num_nodes, len(q), time, file_name_mot)